# Tamban thesis: consultation evidence notebook
**Prepared 15 October 2026.** Use your current thesis source ZIP, not a prompt-package ZIP.

This notebook produces an evidence ZIP for a professor consultation. It does **not** retrain or rewrite your system. It keeps four evidence levels separate:

- **Measured from files:** dataset counts, annotation geometry, checkpoint availability, source syntax.
- **Exploratory:** brightness/saturation/highlight/gradient measurements on training Body ROIs only.
- **Synthetic:** controlled inputs to your actual pure grading policy, when you explicitly allow repository code execution.
- **Optional measured evidence:** expert-labeled fish metrics, logs from your inspection computer, and benchmarks of actual supplied checkpoints.

**Not automatically measured:** live webcam FPS, whole-system accuracy, specimen independence, or expert approval. Empty templates and absent models remain blocked, not zero-error results.

Only upload research files you are authorized to use. Exclude credentials, `.env` files, consent signatures, and consultation recordings from the source ZIP used here.

## 1. Install small analysis dependencies
No GPU or neural model is needed for the quick run. Do not install the entire application just to obtain these tables.

In [ ]:
import importlib.util, subprocess, sys
needed = {'numpy':'numpy>=1.24,<3', 'pandas':'pandas>=2,<3',
          'PIL':'Pillow>=10,<12', 'matplotlib':'matplotlib>=3.9,<4', 'yaml':'PyYAML>=6,<7'}
missing = [requirement for module, requirement in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Analysis dependencies are available.')

## 2. Write the included analysis module
This cell contains the complete helper; you do not need to upload a separate Python file.

In [ ]:
%%writefile /content/consultation_evidence.py
"""Consultation evidence tools; no retraining or source-code modification.

Default run: inventories, COCO audit, TRAIN-only appearance exploration.
Optional: execute the trusted pure policy; score user-supplied expert labels;
read local timing logs; benchmark explicitly supplied trusted checkpoints.
All outputs describe their evidence level. Python 3.10+.
"""
from __future__ import annotations
import argparse
import ast
import dataclasses
import hashlib
import importlib.util
import importlib.metadata
import json
import math
import platform
import re
import shutil
import stat
import sys
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

GRADES = ['Class A', 'Class B', 'Class C', 'Rejected']
PARTS = ['Body', 'Head', 'Tail']
SPLITS = ['train', 'validation', 'test']


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def clean_json(obj):
    if isinstance(obj, dict):
        return {str(k): clean_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        return float(obj) if math.isfinite(obj) else None
    if isinstance(obj, Path):
        return str(obj)
    return obj


def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(clean_json(obj), indent=2, allow_nan=False), encoding='utf-8')


def table(rows, path, columns=None):
    df = pd.DataFrame(rows, columns=columns)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return df


def save_plot(path, title, ylabel=None):
    plt.title(title)
    if ylabel:
        plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.close()


def safe_extract(archive, dest, max_bytes=12 * 1024**3, max_members=100000):
    """Reject traversal, symlinks, duplicate paths, and oversized archives."""
    dest = Path(dest).resolve()
    if dest.exists() and any(dest.iterdir()):
        raise ValueError('Extraction directory must be empty; source files are never overwritten.')
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        members = z.infolist()
        if len(members) > max_members or sum(i.file_size for i in members) > max_bytes:
            raise ValueError('Archive exceeds safety limits.')
        used = set()
        for info in members:
            name = info.filename.replace('\\', '/')
            p = PurePosixPath(name)
            target = (dest / name).resolve()
            if p.is_absolute() or '..' in p.parts or ':' in name or not target.is_relative_to(dest):
                raise ValueError(f'Unsafe archive path: {name}')
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f'Symlink rejected: {name}')
            if str(target) in used:
                raise ValueError(f'Duplicate archive target: {name}')
            used.add(str(target))
        for info in members:
            name = info.filename.replace('\\', '/')
            target = dest / name
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst, 1024 * 1024)
    return dest


def find_repo(root):
    hits = sorted(Path(root).rglob('src/inference/grading_policy.py'))
    if len(hits) == 1:
        return hits[0].parents[2]
    if len(hits) > 1:
        raise ValueError('Multiple repositories found. Supply one clean project ZIP.')
    return Path(root)


def environment():
    versions = {}
    for pkg in ['numpy', 'pandas', 'matplotlib', 'Pillow', 'PyYAML', 'torch', 'ultralytics']:
        try:
            versions[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            versions[pkg] = 'not installed'
    return {'created_utc': utc_now(), 'python': sys.version, 'platform': platform.platform(),
            'processor': platform.processor(), 'packages': versions,
            'scope': 'This execution environment; not automatically the inspection computer.'}


def parse_class(label):
    s = str(label).strip().lower().replace('-', '_').replace(' ', '_')
    part = next((p for p in PARTS if p.lower() in s.split('_')), None)
    if 'reject' in s:
        return 'Rejected', part
    match = re.search(r'(?:^|_)([abc])(?:_|$)', s)
    return (f'Class {match.group(1).upper()}' if match else None), part


def split_name(folder):
    return {'valid': 'validation', 'val': 'validation'}.get(folder.lower(), folder.lower())


def original_group(info):
    extra = info.get('extra') or {}
    name = extra.get('name') if isinstance(extra, dict) else None
    return str(name or re.sub(r'\.rf\.[^.]+', '', Path(info['file_name']).name))


def resolve_image(annotation_file, name, repo):
    # Do not silently pick another file with the same basename.
    name = str(name).replace('\\', '/')
    if PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts or ':' in name:
        return None
    for base in (annotation_file.parent, annotation_file.parent / 'images'):
        candidate = (base / name).resolve()
        if candidate.is_relative_to(repo.resolve()) and candidate.is_file():
            return candidate
    return None


def polygon_is_box(segmentation, tolerance=0.02):
    """Geometric box-like heuristic, not an assertion about annotation intent."""
    if not isinstance(segmentation, list) or len(segmentation) != 1:
        return False
    coords = np.asarray(segmentation[0], dtype=float)
    if coords.ndim != 1 or len(coords) < 8 or len(coords) % 2 or not np.isfinite(coords).all():
        return False
    xy = coords.reshape(-1, 2)
    if np.allclose(xy[0], xy[-1]):
        xy = xy[:-1]
    if len(xy) != 4:
        return False
    lo, hi = xy.min(axis=0), xy.max(axis=0)
    if np.any(hi <= lo):
        return False
    normalized = (xy - lo) / (hi - lo)
    corners = {(int(round(x)), int(round(y))) for x, y in normalized}
    return corners == {(0, 0), (1, 0), (1, 1), (0, 1)} and bool(
        np.all(np.abs(normalized - np.round(normalized)) <= tolerance))


def inventory(repo, out):
    rows, syntax = [], []
    for p in sorted(repo.rglob('*')):
        if not p.is_file():
            continue
        rel = str(p.relative_to(repo))
        rows.append({'path': rel, 'suffix': p.suffix.lower(), 'size_bytes': p.stat().st_size})
        if p.suffix == '.py' and p.relative_to(repo).parts[0] in ('src', 'tests'):
            try:
                ast.parse(p.read_text(encoding='utf-8-sig'), filename=rel)
                status, error = 'PARSES', ''
            except Exception as exc:
                status, error = 'ERROR', str(exc)
            syntax.append({'path': rel, 'status': status, 'error': error})
    table(rows, out / 'repository_inventory.csv')
    table(syntax, out / 'python_syntax_check.csv', ['path', 'status', 'error'])
    reg_path = repo / 'models/MODEL_REGISTRY.json'
    models = []
    if reg_path.exists():
        data = json.loads(reg_path.read_text(encoding='utf-8-sig'))
        for item in data.get('models', []):
            p = (repo / item.get('path', '')).resolve()
            exists = p.is_relative_to(repo.resolve()) and p.is_file()
            actual = sha256(p) if exists else None
            expected = item.get('sha256')
            models.append({'id': item.get('id'), 'role': item.get('role'), 'path': item.get('path'),
                           'input_size': item.get('input_size'), 'present_in_archive': exists,
                           'expected_sha256': expected, 'actual_sha256': actual,
                           'checksum_matches': (actual == expected) if exists and expected else None})
    table(models, out / 'checkpoint_inventory.csv')
    config_path = repo / 'configs/grading_engine.yaml'
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8-sig')) if config_path.exists() else {}
    save_json(out / 'grading_config_snapshot.json', cfg)
    provenance = []
    for rel in ['configs/grading_engine.yaml', 'src/inference/grading_policy.py',
                'src/inference/grading_engine.py', 'src/api/camera.py', 'src/api/app.py',
                'src/features/color.py', 'src/features/texture.py', 'models/MODEL_REGISTRY.json']:
        p = repo / rel
        if p.exists():
            provenance.append({'path': rel, 'sha256': sha256(p)})
    table(provenance, out / 'source_fingerprints.csv')
    # Historical artifact inventory is NOT a new evaluation.
    saved = []
    for p in sorted((repo / 'results').rglob('*')) if (repo / 'results').exists() else []:
        if p.is_file() and p.suffix in ('.json', '.csv', '.txt', '.md'):
            saved.append({'path': str(p.relative_to(repo)), 'sha256': sha256(p),
                          'evidence_status': 'SAVED_ARTIFACT_NOT_RERUN'})
    table(saved, out / 'saved_artifacts_index.csv')
    return {'file_count': len(rows), 'syntax_files': len(syntax),
            'syntax_errors': sum(r['status'] == 'ERROR' for r in syntax),
            'registered_checkpoints': len(models),
            'registered_checkpoints_present': sum(r['present_in_archive'] for r in models), 'config': cfg}


def audit_coco(repo, out):
    annotations, image_rows, split_rows, categories, issues = [], [], [], [], []
    for p in sorted(repo.rglob('_annotations.coco.json')):
        d = json.loads(p.read_text(encoding='utf-8-sig'))
        dataset = str(p.parent.parent.relative_to(repo))
        split = split_name(p.parent.name)
        cats = {x['id']: x['name'] for x in d.get('categories', [])}
        images = {x['id']: x for x in d.get('images', [])}
        if len(images) != len(d.get('images', [])):
            issues.append({'file': str(p.relative_to(repo)), 'issue': 'duplicate image IDs'})
        ann_ids = [a['id'] for a in d.get('annotations', [])]
        if len(set(ann_ids)) != len(ann_ids):
            issues.append({'file': str(p.relative_to(repo)), 'issue': 'duplicate annotation IDs'})
        for cid, label in cats.items():
            grade, part = parse_class(label)
            categories.append({'dataset': dataset, 'split': split, 'category_id': cid,
                               'label': label, 'grade': grade, 'part': part})
        for iid, info in images.items():
            image_path = resolve_image(p, info['file_name'], repo)
            image_rows.append({'dataset': dataset, 'split': split, 'image_id': iid,
                               'file_name': info['file_name'], 'source_group_heuristic': original_group(info),
                               'exists': image_path is not None,
                               'sha256': sha256(image_path) if image_path else '',
                               'path': str(image_path.relative_to(repo)) if image_path else '',
                               'width': info.get('width'), 'height': info.get('height')})
        boxlike, polygon_count, used_categories = 0, 0, Counter()
        for a in d.get('annotations', []):
            name = cats.get(a.get('category_id'), 'UNKNOWN')
            grade, part = parse_class(name)
            info = images.get(a.get('image_id'))
            seg = a.get('segmentation')
            is_polygon = isinstance(seg, list) and bool(seg)
            is_box = polygon_is_box(seg) if is_polygon else False
            boxlike += int(is_box)
            polygon_count += int(is_polygon)
            used_categories[name] += 1
            bbox = a.get('bbox', [])
            valid = isinstance(bbox, list) and len(bbox) == 4 and all(
                isinstance(v, (int, float)) and math.isfinite(v) for v in bbox)
            valid = valid and bbox[2] > 0 and bbox[3] > 0
            if info is None or a.get('category_id') not in cats or not valid:
                issues.append({'file': str(p.relative_to(repo)), 'issue': f'Invalid reference/box on annotation {a.get("id")}'})
                continue
            x, y, w, h = bbox
            if x < -1 or y < -1 or x + w > info['width'] + 1 or y + h > info['height'] + 1:
                issues.append({'file': str(p.relative_to(repo)), 'issue': f'Out-of-image box on annotation {a.get("id")}'})
            annotations.append({'dataset': dataset, 'split': split, 'annotation_id': a['id'],
                                'image_id': a['image_id'], 'category': name, 'grade': grade, 'part': part,
                                'boxlike_polygon': is_box, 'is_polygon': is_polygon,
                                '_annotation': a, '_image': info, '_file': p})
        split_rows.append({'dataset': dataset, 'split': split, 'images': len(images),
                           'annotations': len(d.get('annotations', [])),
                           'polygon_annotations': polygon_count, 'boxlike_polygons': boxlike,
                           'boxlike_fraction_of_polygons': boxlike / polygon_count if polygon_count else None})
    splits = table(split_rows, out / 'dataset_split_summary.csv',
                   ['dataset', 'split', 'images', 'annotations', 'polygon_annotations', 'boxlike_polygons', 'boxlike_fraction_of_polygons'])
    images_df = table(image_rows, out / 'dataset_images.csv')
    table(categories, out / 'category_mapping.csv')
    table(issues, out / 'annotation_issues.csv', ['file', 'issue'])
    annotation_export = [{k: v for k, v in a.items() if not k.startswith('_')} for a in annotations]
    table(annotation_export, out / 'annotation_inventory.csv')
    counts = []
    for r in split_rows:
        group = [a for a in annotations if a['dataset'] == r['dataset'] and a['split'] == r['split']]
        for grade in GRADES:
            counts.append({'dataset': r['dataset'], 'split': r['split'], 'grade': grade,
                           'part_annotations': sum(a['grade'] == grade for a in group),
                           'images_with_grade': len({a['image_id'] for a in group if a['grade'] == grade}),
                           'body_annotations': sum(a['grade'] == grade and a['part'] == 'Body' for a in group)})
    cdf = table(counts, out / 'grade_support_by_split.csv')
    cross = []
    if len(images_df):
        for key, description in [('sha256', 'exact_bytes'), ('source_group_heuristic', 'filename_or_extra_name_heuristic')]:
            for (dataset, val), group in images_df[images_df[key] != ''].groupby(['dataset', key]):
                if group['split'].nunique() > 1:
                    cross.append({'dataset': dataset, 'check': description, 'key': val,
                                  'splits': '|'.join(sorted(group['split'].unique())), 'image_count': len(group)})
    table(cross, out / 'cross_split_flags.csv', ['dataset', 'check', 'key', 'splits', 'image_count'])
    for idx, dataset in enumerate(sorted(set(r['dataset'] for r in split_rows)), start=1):
        sub = splits[splits.dataset == dataset].set_index('split').reindex(SPLITS).dropna(subset=['images'])
        plt.figure(figsize=(7.5, 4.2)); plt.bar(sub.index, sub.images)
        save_plot(out / 'charts' / f'dataset_{idx}_image_split_counts.png', f'Dataset {idx}: image records by split', 'Images (not fish)')
        for split in SPLITS:
            g = cdf[(cdf.dataset == dataset) & (cdf.split == split)]
            if g.empty:
                continue
            plt.figure(figsize=(7.5, 4.2)); bars = plt.bar(g.grade, g.part_annotations)
            plt.bar_label(bars, padding=3)
            save_plot(out / 'charts' / f'dataset_{idx}_{split}_grade_support.png', f'Dataset {idx}: {split} part-label support', 'Body/head/tail annotations (not whole fish)')
    return annotations, {'split_records': split_rows, 'annotation_issues': len(issues),
                         'cross_split_flags': len(cross), 'grade_support': counts,
                         'warning': 'Part annotations are not physical-fish counts. No specimen independence is certified.'}


def explore_appearance(repo, annotations, out, per_class=40, seed=42, dataset=None):
    """TRAIN-only exploratory ROI measurements, never whole-fish grade truth."""
    if per_class < 1:
        return {'status': 'NOT_RUN', 'reason': 'per_class is zero'}
    bodies = [a for a in annotations if a['split'] == 'train' and a['part'] == 'Body' and a['grade'] in GRADES]
    if dataset:
        bodies = [a for a in bodies if a['dataset'] == dataset]
    datasets = Counter(a['dataset'] for a in bodies)
    if not datasets:
        return {'status': 'BLOCKED', 'reason': 'No recognized COCO training Body labels.'}
    chosen = dataset or datasets.most_common(1)[0][0]
    bodies = [a for a in bodies if a['dataset'] == chosen]
    rng = np.random.default_rng(seed)
    rows, errors = [], []
    for grade in GRADES:
        candidates = [a for a in bodies if a['grade'] == grade]
        rng.shuffle(candidates)
        used = set()
        for rec in candidates:
            group = original_group(rec['_image'])
            if group in used or len(used) >= per_class:
                continue
            p = resolve_image(rec['_file'], rec['_image']['file_name'], repo)
            if p is None:
                errors.append({'annotation_id': rec['annotation_id'], 'error': 'missing image'}); continue
            try:
                with Image.open(p) as im:
                    if im.width * im.height > 25000000:
                        raise ValueError('Image too large for quick exploration')
                    rgb = np.asarray(im.convert('RGB'), dtype=np.float64) / 255.0
                h, w = rgb.shape[:2]
                seg = rec['_annotation'].get('segmentation')
                mask_image = Image.new('1', (w, h), 0)
                draw = ImageDraw.Draw(mask_image)
                if isinstance(seg, list) and seg:
                    for polygon in seg:
                        if len(polygon) >= 6 and len(polygon) % 2 == 0:
                            draw.polygon(list(zip(polygon[::2], polygon[1::2])), fill=1)
                    mask_kind = 'boxlike_polygon_ROI' if rec['boxlike_polygon'] else 'polygon_ROI_unverified_outline'
                else:
                    x, y, bw, bh = rec['_annotation']['bbox']
                    draw.rectangle((x, y, x + bw, y + bh), fill=1)
                    mask_kind = 'bbox_ROI'
                mask = np.asarray(mask_image, dtype=bool)
                if mask.sum() < 16:
                    raise ValueError('Too few ROI pixels')
                maximum = rgb.max(axis=2)
                chroma = maximum - rgb.min(axis=2)
                sat = np.divide(chroma, maximum, out=np.zeros_like(maximum), where=maximum > 0)
                gray = rgb @ np.array([0.2126, 0.7152, 0.0722])
                gy, gx = np.gradient(gray)
                interior = mask.copy()
                interior[1:] &= mask[:-1]; interior[:-1] &= mask[1:]
                interior[:, 1:] &= mask[:, :-1]; interior[:, :-1] &= mask[:, 1:]
                grad = np.hypot(gx, gy)
                rows.append({'dataset': chosen, 'split': 'train', 'grade_of_body_annotation': grade,
                             'image_path': str(p.relative_to(repo)), 'source_group_heuristic': group,
                             'annotation_id': rec['annotation_id'], 'ROI_kind': mask_kind,
                             'roi_pixels': int(mask.sum()), 'mean_value': float(maximum[mask].mean()),
                             'brightness_p95': float(np.percentile(maximum[mask], 95)),
                             'mean_saturation': float(sat[mask].mean()),
                             'highlight_ratio_proxy': float(((maximum >= .90) & (sat <= .20))[mask].mean()),
                             'mean_gradient_proxy': float(grad[interior].mean()) if interior.any() else None,
                             'measurement_scope': 'exploratory body ROI; not a live-grading input'})
                used.add(group)
            except Exception as exc:
                errors.append({'annotation_id': rec['annotation_id'], 'error': str(exc)})
    df = table(rows, out / 'training_body_roi_features.csv')
    table(errors, out / 'feature_extraction_errors.csv', ['annotation_id', 'error'])
    if not len(df):
        return {'status': 'BLOCKED', 'reason': 'No usable body ROI measurements.'}
    for feature, title in [('mean_value', 'Brightness (HSV Value)'),
                           ('mean_saturation', 'Saturation'), ('highlight_ratio_proxy', 'Highlight-ratio proxy'),
                           ('mean_gradient_proxy', 'Surface-gradient proxy')]:
        groups = [g for g in GRADES if (df.grade_of_body_annotation == g).any()]
        values = [df.loc[df.grade_of_body_annotation == g, feature].dropna().to_numpy() for g in groups]
        plt.figure(figsize=(8, 4.5))
        tick_labels = [f'{g}\nn={len(v)}' for g, v in zip(groups, values)]
        try:
            plt.boxplot(values, tick_labels=tick_labels, showfliers=True)
        except TypeError:
            # Compatibility with existing Matplotlib <3.9 in a notebook runtime.
            plt.boxplot(values, labels=tick_labels, showfliers=True)
        save_plot(out / 'charts' / f'train_body_{feature}.png', f'{title}: training Body annotations only', 'Descriptive value; not grade confidence')
    plt.figure(figsize=(7, 5))
    for g in ['Class B', 'Class C']:
        sub = df[df.grade_of_body_annotation == g]
        if len(sub):
            plt.scatter(sub.mean_value, sub.mean_saturation, label=f'{g} (n={len(sub)})', alpha=.65)
    plt.xlabel('Mean HSV Value'); plt.legend()
    save_plot(out / 'charts' / 'train_BC_brightness_saturation.png', 'B/C training Body ROIs: exploratory comparison', 'Mean saturation')
    summary = df.groupby('grade_of_body_annotation')[['mean_value', 'mean_saturation', 'highlight_ratio_proxy']].agg(['count', 'mean', 'median', 'std'])
    summary.to_csv(out / 'training_body_feature_summary.csv')
    return {'status': 'MEASURED_EXPLORATORY', 'dataset': chosen, 'samples': len(df),
            'seed': seed, 'per_class_limit': per_class, 'counts': df.grade_of_body_annotation.value_counts().to_dict(),
            'caution': 'One body ROI per filename-source group per grade; physical specimen identity unknown. Rectangle ROIs may contain background. Not a validation experiment.'}


def load_pure_policy(repo):
    path = repo / 'src/inference/grading_policy.py'
    name = '_consultation_policy_' + sha256(path)[:12]
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def policy_from_config(mod, cfg):
    g = cfg.get('grading', {})
    allowed = {f.name for f in dataclasses.fields(mod.GradingPolicy)}
    args = {k: v for k, v in g.items() if k in allowed}
    for part, value in g.get('weights', {}).items():
        args[f'{part}_weight'] = value
    override = g.get('rejected_override', {})
    args['rejected_override_threshold'] = override.get('threshold') if override.get('enabled') else None
    args['maximum_grade_stddev'] = g.get('temporal', {}).get('maximum_grade_stddev')
    return mod.GradingPolicy(**args)


def run_policy_probes(repo, cfg, out, trusted=False):
    if not trusted:
        return {'status': 'BLOCKED', 'reason': 'Repository execution not authorized; set TRUST_REPOSITORY_CODE=True for your own trusted repository.'}
    mod = load_pure_policy(repo)
    base = policy_from_config(mod, cfg)
    policies = {'CURRENT_CONFIG': base,
                'PROPOSED_TWO_PART_FIXED_WEIGHTS': dataclasses.replace(base, grading_mode='standard', require_body=True,
                    minimum_regions_observed=2, minimum_original_weight_coverage=.70),
                'STRICT_THREE_PART_REFERENCE': dataclasses.replace(base, grading_mode='strict',
                    strict_minimum_regions_observed=3, strict_minimum_original_weight_coverage=1.0)}
    cases = [
        ('All parts A', {'Body': ('Class A', .87), 'Head': ('Class A', .79), 'Tail': ('Class A', .75)}, {}),
        ('Body + Head A, tail unobserved', {'Body': ('Class A', .87), 'Head': ('Class A', .79)}, {}),
        ('Body + Tail A, head unobserved', {'Body': ('Class A', .87), 'Tail': ('Class A', .79)}, {}),
        ('Body only A at 1.00', {'Body': ('Class A', 1.0)}, {}),
        ('Body only A at 0.87', {'Body': ('Class A', .87)}, {}),
        ('Head + Tail only A', {'Head': ('Class A', .90), 'Tail': ('Class A', .90)}, {}),
        ('All parts Rejected', {'Body': ('Rejected', .90), 'Head': ('Rejected', .90), 'Tail': ('Rejected', .90)}, {}),
        ('Exact A/B tie', {'Body': ('Class A', 1.0), 'Head': ('Class B', 1.0), 'Tail': ('Class B', 1.0)}, {}),
        ('Ambiguous ownership', {'Body': ('Class A', .90), 'Head': ('Class A', .90)}, {'association_status': 'ambiguous'}),
        ('Temporal evidence not ready', {'Body': ('Class A', .90), 'Head': ('Class A', .90)}, {'temporal_ready': False}),
        ('No parts, model unavailable', {}, {'model2_available': False}),
    ]
    rows = []
    for case, evidence, context in cases:
        for policy_name, policy in policies.items():
            args = {'part_scores': {p: {g: c} for p, (g, c) in evidence.items()},
                    'presence_confidence': {p: c for p, (_, c) in evidence.items()},
                    'observation_count': 2, 'temporal_ready': True}
            args.update(context)
            v = mod.grade_fish(mod.FishGradingInput(**args), policy).to_dict()
            coverage = v['evidence_coverage']
            rows.append({'scenario': case, 'policy': policy_name, 'final_grade': v['final_grade'],
                         'fixed_weight_top_support': v['top_support'], 'coverage': coverage,
                         'hypothetical_normalized_top_support_NOT_RUNTIME': (v['top_support'] / coverage) if coverage else None,
                         'reason_codes': '|'.join(v['reason_codes']),
                         'evidence_type': 'SYNTHETIC_POLICY_TEST_NOT_ACCURACY'})
    table(rows, out / 'policy_synthetic_scenarios.csv')
    (out / 'policy_probe_notes.md').write_text(
        '# Synthetic policy checks\n\nThese calls execute the actual pure grading policy, not either neural model, tracking, or temporal aggregation. '
        'Readiness is injected as true except the explicitly unready scenario. '
        'The proposed and strict policies are in-memory alternatives; no YAML is modified. '
        'Normalized support is arithmetic only, not an implemented alternative verdict. '
        'There is no physical-missing-head input here: unobserved and severed cannot be distinguished by these scores.\n', encoding='utf-8')
    return {'status': 'EXECUTED_SYNTHETIC', 'scenarios': len(cases), 'policy_variants': len(policies),
            'source': 'src/inference/grading_policy.py', 'source_sha256': sha256(repo / 'src/inference/grading_policy.py')}


def normalize_grade(s, allow_abstain=False):
    s = str(s).strip()
    alias = {'A': 'Class A', 'B': 'Class B', 'C': 'Class C', 'Rejected': 'Rejected',
             'Ungraded': 'Ungraded', 'Undetected': 'Undetected'}
    result = alias.get(s, s)
    allowed = GRADES + (['Ungraded', 'Undetected'] if allow_abstain else [])
    if result not in allowed:
        raise ValueError(f'Unrecognized grade: {s!r}; do not silently discard this row.')
    return result


def evaluate_fish_csv(truth_path, pred_path, out, selected_split='validation'):
    """One explicit, manually matched case_id per fish; no label inference from parts."""
    truth, pred = pd.read_csv(truth_path, dtype=str).fillna(''), pd.read_csv(pred_path, dtype=str).fillna('')
    required_t = {'case_id', 'specimen_id', 'split', 'true_grade', 'expert_id'}
    required_p = {'case_id', 'run_id', 'predicted_grade', 'model1_id', 'model2_id', 'policy_id'}
    if not required_t.issubset(truth) or not required_p.issubset(pred):
        raise ValueError('CSV headers do not match templates. Use separate expert truth and predictions.')
    if truth.empty or pred.empty:
        raise ValueError('Templates are empty; no accuracy can be calculated.')
    if truth.case_id.eq('').any() or truth.case_id.duplicated().any():
        raise ValueError('Ground truth requires unique, nonempty case_id values.')
    if pred.case_id.eq('').any() or pred.run_id.eq('').any() or pred.duplicated(['run_id', 'case_id']).any():
        raise ValueError('Predictions require unique (run_id, case_id) pairs.')
    for col in ['specimen_id', 'expert_id']:
        if truth[col].eq('').any():
            raise ValueError(f'Ground-truth {col} is required for traceability.')
    truth['split'] = truth['split'].map(split_name)
    leaking = truth.groupby('specimen_id')['split'].nunique()
    if (leaking > 1).any():
        raise ValueError('A specimen_id crosses splits. Repair provenance before scoring.')
    truth.true_grade = truth.true_grade.map(normalize_grade)
    pred.predicted_grade = pred.predicted_grade.map(lambda x: normalize_grade(x, True))
    chosen = truth[truth.split == selected_split].copy()
    if chosen.empty:
        raise ValueError(f'No expert labels for split {selected_split!r}.')
    summaries, class_rows = [], []
    pred_cols = GRADES + ['Ungraded', 'Undetected']
    for run_id, group in pred.groupby('run_id'):
        joined = chosen.merge(group, on='case_id', how='left', validate='one_to_one')
        joined['predicted_grade'] = joined.predicted_grade.fillna('Undetected')
        table(joined.to_dict('records'), out / f'evaluation_joined_{slug(run_id)}.csv')
        unknown = group[~group.case_id.isin(truth.case_id)]
        table(unknown.to_dict('records'), out / f'unmatched_predictions_{slug(run_id)}.csv', list(pred.columns))
        cm = np.array([[int(((joined.true_grade == g) & (joined.predicted_grade == p)).sum()) for p in pred_cols] for g in GRADES])
        n = len(joined); graded = joined.predicted_grade.isin(GRADES)
        correct = int((joined.true_grade == joined.predicted_grade).sum())
        f1s = []
        for i, grade in enumerate(GRADES):
            tp, support = int(cm[i, i]), int(cm[i].sum())
            predicted = int(cm[:, i].sum())
            precision = tp / predicted if predicted else None
            recall = tp / support if support else None
            f1 = 2 * tp / (support + predicted) if support + predicted else None
            if support:
                f1s.append(f1)
            class_rows.append({'run_id': run_id, 'grade': grade, 'support': support,
                               'precision_matched_cases_only': precision, 'recall': recall, 'f1': f1})
        summary = {'run_id': run_id, 'split': selected_split, 'expert_labeled_fish': n,
                   'graded': int(graded.sum()), 'ungraded': int((joined.predicted_grade == 'Ungraded').sum()),
                   'undetected_or_missing_prediction': int((joined.predicted_grade == 'Undetected').sum()),
                   'grade_coverage': float(graded.mean()), 'correct_grade_rate_all_GT_fish': correct / n,
                   'accuracy_on_graded_fish_only': correct / int(graded.sum()) if graded.any() else None,
                   'macro_f1_four_grades': float(np.mean(f1s)) if len(f1s) == 4 else None,
                   'macro_f1_supported_grades_only': float(np.mean(f1s)),
                   'unmatched_prediction_rows': len(unknown),
                   'caution': 'No certification of independent specimens, complete scene truth, detection mAP, or counting accuracy.'}
        summaries.append(summary)
        pd.DataFrame(cm, index=GRADES, columns=pred_cols).to_csv(out / f'confusion_{slug(run_id)}.csv')
        plt.figure(figsize=(9, 5.2)); plt.imshow(cm, aspect='auto'); plt.colorbar(label='Expert-labeled fish')
        plt.xticks(range(len(pred_cols)), pred_cols, rotation=25, ha='right'); plt.yticks(range(4), GRADES)
        for i in range(4):
            for j in range(len(pred_cols)):
                plt.text(j, i, str(cm[i, j]), ha='center', va='center')
        plt.xlabel('System decision (missing predictions included)')
        save_plot(out / 'charts' / f'confusion_{slug(run_id)}.png', f'{run_id}: {selected_split}, n={n}', 'Expert grade')
    table(summaries, out / 'fish_grade_metrics.csv'); table(class_rows, out / 'fish_grade_per_class.csv')
    save_json(out / 'fish_evaluation_provenance.json', {'truth_sha256': sha256(truth_path), 'prediction_sha256': sha256(pred_path),
              'scope': 'Declared expert labels; not independently verified by this program.',
              'metric_note': 'Precision is on matched GT cases only; unmatched prediction rows are listed separately, not silently included/excluded as true negatives.'})
    return summaries


def slug(s):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', str(s))[:80]


def analyze_runtime_log(log_path, out):
    samples = []
    for line in Path(log_path).read_text(encoding='utf-8').splitlines():
        if line.strip():
            samples.append(json.loads(line))
    valid = [r for r in samples if r.get('record_type') == 'sample' and r.get('ok')]
    if len(valid) < 2:
        raise ValueError('Need at least two successful local runtime samples. No local benchmark inferred.')
    rows, stage_rows = [], []
    for prev, cur in zip(valid, valid[1:]):
        dt = cur['elapsed_s'] - prev['elapsed_s']
        if dt <= 0:
            continue
        counters = cur['runtime'].get('performance', {}).get('counters', {})
        old = prev['runtime'].get('performance', {}).get('counters', {})
        row = {'elapsed_s': cur['elapsed_s'], 'poll_latency_ms': cur['request_ms'], 'debug_enabled': cur['runtime'].get('enabled'),
               'active_tracks': cur['runtime'].get('active_tracks')}
        for key, label in [('camera_capture', 'capture_fps'), ('processed_frame', 'published_processed_fps'),
                           ('model1', 'model1_calls_s'), ('model2', 'model2_calls_s')]:
            delta = counters.get(key, 0) - old.get(key, 0)
            row[label] = delta / dt if key in counters and key in old and delta >= 0 else None
        rows.append(row)
    # Use the final rolling-window summary. Never average percentiles or add nested timers.
    timing = valid[-1]['runtime'].get('performance', {}).get('timings', {})
    for stage, stats in timing.items():
        stage_rows.append({'stage': stage, **stats, 'scope': 'last server rolling-window summary; timers can overlap'})
    df = table(rows, out / 'local_runtime_rates.csv')
    sdf = table(stage_rows, out / 'local_runtime_last_window.csv')
    plt.figure(figsize=(8, 4.5))
    for c in ['capture_fps', 'published_processed_fps']:
        if c in df:
            plt.plot(df.elapsed_s, df[c], label=c)
    plt.xlabel('Elapsed collector time (seconds)'); plt.legend()
    save_plot(out / 'charts' / 'local_capture_vs_processed_fps.png', 'Actual local API counters (not browser display FPS)', 'Frames per second over sample intervals')
    if len(sdf) and 'p50' in sdf:
        view = sdf.dropna(subset=['p50']).sort_values('p50')
        plt.figure(figsize=(9, max(4, len(view) * .32)))
        plt.barh(view.stage, view.p50); plt.xlabel('Reported p50 milliseconds')
        save_plot(out / 'charts' / 'local_stage_timing_p50.png', 'Final rolling-window stage times: nested, NOT additive')
    summary = {'status': 'LOCAL_COUNTER_OBSERVATIONS', 'successful_polls': len(valid),
               'counter_window_seconds': valid[-1]['elapsed_s'] - valid[0]['elapsed_s'],
               'any_debug_enabled': any(bool(v['runtime'].get('enabled')) for v in valid),
               'warning': 'Published processed FPS is not browser FPS. Model2 timing may repeat per parent. Stage timers are overlapping and are not summed.'}
    for c in ['capture_fps', 'published_processed_fps']:
        summary[c + '_interval_median'] = df[c].median() if c in df else None
    save_json(out / 'local_runtime_summary.json', summary)
    return summary


def benchmark_checkpoint(checkpoint, image_paths, out, label, imgsz=768, device='cpu',
                         half=False, warmup=3, repeats=30, conf=.25, iou=.7):
    """Real checkpoint component timing; NOT the full application or its accuracy."""
    checkpoint = Path(checkpoint)
    if not checkpoint.is_file():
        raise FileNotFoundError('Supply the actual trusted checkpoint; no pretrained model is substituted.')
    if repeats < 3 or warmup < 1:
        raise ValueError('Use at least 3 measured runs and 1 warm-up.')
    import torch
    from ultralytics import YOLO
    gpu = str(device).lower() not in ('cpu', 'mps')
    if gpu and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable; choose CPU explicitly, do not label fallback as GPU.')
    if half and not gpu:
        raise ValueError('This helper permits FP16 only with CUDA. Default FP32 is safer.')
    arrays, decode_rows = [], []
    for p in list(image_paths)[:12]:
        t = time.perf_counter()
        with Image.open(p) as im:
            rgb = np.asarray(im.convert('RGB'))
        # Ultralytics NumPy input convention is BGR.
        arrays.append(np.ascontiguousarray(rgb[:, :, ::-1]))
        decode_rows.append({'path': str(p), 'image_sha256': sha256(p), 'decode_ms': (time.perf_counter() - t) * 1000})
    if not arrays:
        raise ValueError('No usable input images supplied.')
    t = time.perf_counter(); model = YOLO(str(checkpoint)); load_ms = (time.perf_counter() - t) * 1000
    gpu_index = int(str(device).replace('cuda:', '')) if gpu else None
    def sync():
        if gpu:
            torch.cuda.synchronize(gpu_index)
    def one(image):
        sync(); start = time.perf_counter()
        results = model.predict(source=image, imgsz=imgsz, device=device, half=half,
                                conf=conf, iou=iou, verbose=False, save=False)
        count = 0
        if results and results[0].boxes is not None:
            count = len(results[0].boxes.data.cpu().numpy())
        sync()
        return (time.perf_counter() - start) * 1000, count
    cold_ms, _ = one(arrays[0])
    for i in range(warmup):
        one(arrays[i % len(arrays)])
    rows = []
    for i in range(repeats):
        ms, count = one(arrays[i % len(arrays)])
        rows.append({'iteration': i, 'input_index': i % len(arrays), 'wall_ms': ms, 'detections': count})
    tag = slug(f'{label}_{device}_{imgsz}_{"fp16" if half else "fp32"}')
    frame = table(rows, out / f'component_benchmark_{tag}.csv')
    table(decode_rows, out / f'benchmark_inputs_{tag}.csv')
    summary = {'label': label, 'checkpoint_name': checkpoint.name, 'checkpoint_sha256': sha256(checkpoint),
               'requested_device': str(device), 'actual_device': str(next(model.model.parameters()).device),
               'imgsz': imgsz, 'half': half, 'conf': conf, 'iou': iou, 'load_ms': load_ms, 'cold_first_call_ms': cold_ms,
               'warmup_calls': warmup, 'measured_calls': repeats, 'mean_wall_ms': frame.wall_ms.mean(),
               'p50_wall_ms': frame.wall_ms.median(), 'p95_wall_ms': frame.wall_ms.quantile(.95),
               'component_throughput_fps_equivalent': 1000 / frame.wall_ms.mean(),
               'scope': 'In-memory single-model predict + CPU result materialization only. NOT live camera FPS or accuracy.',
               'environment': environment()}
    save_json(out / f'component_benchmark_{tag}.json', summary)
    return summary


def package_evidence(out):
    out = Path(out)
    return Path(shutil.make_archive(str(out), 'zip', root_dir=out))


def run_quick(archive, output_parent, trusted=False, per_class=40, seed=42):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    base = Path(output_parent) / f'consultation_run_{stamp}'
    if base.exists():
        base = base.with_name(base.name + '_' + str(time.time_ns())[-6:])
    out, extracted = base / 'evidence', base / 'extracted'
    out.mkdir(parents=True); (out / 'charts').mkdir()
    safe_extract(archive, extracted)
    repo = find_repo(extracted)
    save_json(out / 'run_provenance.json', {'zip_name': Path(archive).name, 'zip_sha256': sha256(archive),
              'environment': environment(), 'seed': seed, 'trusted_repository_execution': trusted,
              'source_note': 'The supplied archive snapshot, not a claim about unuploaded local changes.'})
    inv = inventory(repo, out)
    anns, audit = audit_coco(repo, out)
    feature = explore_appearance(repo, anns, out, per_class, seed)
    try:
        policy = run_policy_probes(repo, inv['config'], out, trusted)
    except Exception as exc:
        policy = {'status': 'BLOCKED', 'reason': f'Policy adapter incompatible: {type(exc).__name__}: {exc}'}
    summary = {'inventory': {k: v for k, v in inv.items() if k != 'config'},
               'dataset_audit': audit, 'appearance_exploration': feature, 'policy_probes': policy,
               'real_model_benchmark': 'NOT_RUN: explicit checkpoints and benchmark call required',
               'local_camera_FPS': 'NOT_MEASURED: collect on the inspection computer',
               'fish_grade_accuracy': 'BLOCKED: separate expert fish-level labels and matched predictions required',
               'complete_application_tests': 'NOT_RUN: syntax parsing and synthetic policy calls are not a full test suite'}
    save_json(out / 'summary.json', summary)
    lines = ['# Consultation evidence summary', '', f'Archive: `{Path(archive).name}`',
             f'SHA-256: `{sha256(archive)}`', '', '## What was actually checked',
             f'- {inv["file_count"]} files inventoried; {inv["syntax_files"]} source/test Python files parsed; {inv["syntax_errors"]} syntax errors.',
             f'- {inv["registered_checkpoints_present"]}/{inv["registered_checkpoints"]} registered checkpoint paths have bytes in this archive.',
             f'- {len(audit["split_records"])} COCO split records audited; {audit["cross_split_flags"]} cross-split hash/name flags.',
             f'- Appearance exploration: {feature.get("status")}; {feature.get("samples", 0)} training Body ROIs.',
             f'- Pure-policy probe status: {policy.get("status")}.', '', '## Important interpretation',
             'Dataset label counts describe parts, not independently graded whole fish. Filename groups cannot establish physical specimen independence.',
             'A box-like segmentation polygon is still a region of interest; do not call it a verified fish-only mask.',
             'Appearance charts use training annotations, not model predictions; they do not prove a grading improvement.',
             'Policy examples are synthetic inputs to the actual pure grading function; they do not prove model accuracy or temporal stability.',
             'Old reports in saved_artifacts_index.csv were not rerun. No live FPS, new accuracy, full test-suite result, or expert approval is invented.',
             '', '## Present these first',
             '`dataset_split_summary.csv`, `grade_support_by_split.csv`, the class-support charts, and `policy_synthetic_scenarios.csv` (when enabled).',
             'Use `training_body_roi_features.csv` and its charts only as exploratory feature evidence.',
             'Supply `local_runtime.jsonl` for actual-computer speed evidence; supply completed truth/prediction templates for fish-level metrics.',
             '', '## Questions that these outputs support',
             'How should incomplete parts be handled? Is the proposed two-part policy acceptable? Which classes need more independent examples?',
             'Are the current rectangular annotations sufficient, or are fish-only masks and explicit belly-defect labels needed?',
             'What FPS/decision latency and counting/grade error rates are acceptable on the intended hardware?', '']
    (out / 'PRESENTATION_SUMMARY.md').write_text('\n'.join(lines), encoding='utf-8')
    return {'repo': str(repo), 'output_dir': str(out), 'summary': summary}


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--zip', required=True, help='Your thesis source ZIP, not this consultation package')
    p.add_argument('--out', default='consultation_runs')
    p.add_argument('--trust-repository-code', action='store_true')
    p.add_argument('--per-class', type=int, default=40)
    a = p.parse_args()
    r = run_quick(a.zip, a.out, trusted=a.trust_repository_code, per_class=a.per_class)
    print(json.dumps(r['summary'], indent=2)); print('Evidence ZIP:', package_evidence(r['output_dir']))

if __name__ == '__main__':
    main()


## 3. Upload the thesis source ZIP
Run the cell and select `Thesis-main (2).zip` or your newer source ZIP. Upload only one ZIP in this cell. Keep the browser open while uploading.

For a large ZIP already available in your Colab filesystem, set `ZIP_PATH` to its existing path instead of uploading again.

In [ ]:
from pathlib import Path
from google.colab import files
ZIP_PATH = ""  # Optional existing path, e.g. /content/Thesis-main (2).zip
if not ZIP_PATH:
    uploaded = files.upload()
    candidates = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Upload exactly one thesis source ZIP, not the consultation package.')
    ZIP_PATH = str(candidates[0])
    del uploaded
if not Path(ZIP_PATH).is_file():
    raise FileNotFoundError(ZIP_PATH)
print('Selected:', ZIP_PATH)

## 4. Run the quick evidence collection
`TRUST_REPOSITORY_CODE=False` is the safe default. Set it to `True` only for your own trusted source ZIP to run the actual `grading_policy.py` examples. The default does not execute repository Python code.

Appearance analysis selects at most 40 training Body annotations per class, one per original-filename group. A class may have fewer usable groups. This is not a required scientific sample size or proof of specimen independence.

In [ ]:
import sys
sys.path.insert(0, '/content')
import consultation_evidence as ce
TRUST_REPOSITORY_CODE = False  # Set True only after confirming this is your own trusted repository.
PER_CLASS = 40
run = ce.run_quick(ZIP_PATH, '/content/consultation_runs', trusted=TRUST_REPOSITORY_CODE,
                   per_class=PER_CLASS, seed=42)
OUT = Path(run['output_dir'])
REPO = Path(run['repo'])
print('Output:', OUT)
print((OUT / 'PRESENTATION_SUMMARY.md').read_text())

## 5. View the tables and plots
Read axes and sample counts. Body-part labels are not fish-level grades; rectangular ROIs are not verified fish-only segmentation masks.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
for name in ['dataset_split_summary.csv', 'grade_support_by_split.csv', 'checkpoint_inventory.csv']:
    print('\n' + name)
    display(pd.read_csv(OUT / name))
if (OUT / 'policy_synthetic_scenarios.csv').exists():
    display(pd.read_csv(OUT / 'policy_synthetic_scenarios.csv'))
for p in sorted((OUT / 'charts').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p), width=800))

## 6. Optional: score real expert-assigned whole-fish grades
Skip this section when you do not have labels. Do not use Model 2 predictions as ground truth. Use the CSV headers provided with this package.

`case_id` is a manually matched evaluation identity for a fish, not a guessed relationship between parent and part IDs. Use one ground-truth row per fish and one prediction per `(run_id, case_id)`. Do not count repeated video frames as independent fish.

The evaluator includes missing predictions as **Undetected**, keeps **Ungraded**, reports coverage and conditional accuracy separately, and lists predictions with no matching truth identity. Its precision is explicitly limited to matched cases; complete-scene detection metrics need a separate evaluator.

Use validation data for development. Do not tune on the final test set.

In [ ]:
RUN_FISH_EVALUATION = False
EVALUATION_SPLIT = 'validation'
if RUN_FISH_EVALUATION:
    upload = files.upload()  # Upload completed expert_fish_labels.csv and fish_predictions.csv
    result = ce.evaluate_fish_csv('/content/expert_fish_labels.csv', '/content/fish_predictions.csv',
                                  OUT, selected_split=EVALUATION_SPLIT)
    display(pd.DataFrame(result))
else:
    print('BLOCKED / not requested: no new fish-level accuracy reported.')

## 7. Optional: examine the actual inspection computer's FPS
Run `capture_local_runtime.py` **on the computer running the application**, while an inspection is active, with ordinary debug capture off. Then upload its `local_runtime.jsonl` here.

```powershell
python capture_local_runtime.py --seconds 90 --interval 2 --output local_runtime.jsonl
```

The sampler reads the existing API; it does not enable diagnostics or change models/settings. The plot uses cumulative counter differences. It does not claim browser-display FPS. Last-window stage times can overlap, and Model 2 timing can be recorded more than once per full-frame inference; do not sum them or average p95 values.

In [ ]:
RUN_LOCAL_LOG_ANALYSIS = False
if RUN_LOCAL_LOG_ANALYSIS:
    upload = files.upload()  # Select local_runtime.jsonl
    print(ce.analyze_runtime_log('/content/local_runtime.jsonl', OUT))
else:
    print('NOT MEASURED: actual inspection-computer FPS requires its local log.')

## 8. Optional: real-model component benchmarks
**This is optional and slower.** Supply trusted `.pt` files from your project; the notebook never substitutes a generic pretrained detector. Checkpoint loading can execute serialized code, so use only checkpoints you trust.

Choose a Colab GPU runtime for GPU measurements and verify availability below. Colab hardware is not your inspection computer. These tests measure one model's `predict` call plus result transfer on in-memory images, not tracking, association, camera encoding, UI latency, or final grading accuracy.

The defaults compare 640 and 768 for each checkpoint, use FP32, and keep warm-up separate. Smaller input size is an experiment, not permission to change production settings. Full-frame inputs are used; this does not implement your application's crop-versus-full-frame pipeline.

In [ ]:
RUN_MODEL_BENCHMARKS = False
TRUST_CHECKPOINTS = False
MODEL1_PATH = ''  # Example: /content/model1_fish_parent_detector.pt
MODEL2_PATH = ''  # Example: /content/model2_tamban_yolo26m_partdet_r03_devbest.pt
DEVICE = '0'      # Use 'cpu' explicitly for CPU testing.
INPUT_SIZES = [640, 768]
FP16 = False
REPEATS = 30
if RUN_MODEL_BENCHMARKS:
    if not TRUST_CHECKPOINTS:
        raise ValueError('Confirm TRUST_CHECKPOINTS=True for your own trusted model files.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.4.142,<9'])
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    if not MODEL1_PATH or not MODEL2_PATH:
        print('Upload the actual model checkpoints, then fill both model paths and rerun this cell.')
        files.upload()
        raise ValueError('Set MODEL1_PATH and MODEL2_PATH explicitly; no automatic substitute is allowed.')
    images = pd.read_csv(OUT / 'dataset_images.csv').fillna('')
    samples = images[(images['split'] == 'train') & (images['path'] != '')].head(12)
    image_paths = [REPO / p for p in samples.path]
    results = []
    for label, ckpt, confidence in [('Model1', MODEL1_PATH, .5), ('Model2', MODEL2_PATH, .25)]:
        for size in INPUT_SIZES:
            results.append(ce.benchmark_checkpoint(ckpt, image_paths, OUT, label, imgsz=size,
                            device=DEVICE, half=FP16, warmup=3, repeats=REPEATS, conf=confidence))
    display(pd.DataFrame([{k:v for k,v in r.items() if k != 'environment'} for r in results]))
else:
    print('NOT RUN: optional checkpoint benchmarks. No local-FPS claim is made.')

## 9. Package and download
Rerun this after any optional sections. The download includes evidence tables, graphs, provenance, and interpretation notes, but **not** your source ZIP, model files, or private recordings. It does not automatically establish model accuracy or validate a grading rule.

In [ ]:
ce.save_json(OUT / 'final_notebook_environment.json', ce.environment())
archive = ce.package_evidence(OUT)
print('Ready:', archive)
files.download(str(archive))

## Official implementation references
- Google Colab FAQ: https://research.google.com/colaboratory/faq.html
- Ultralytics prediction arguments and result structures: https://docs.ultralytics.com/modes/predict/
- PyTorch benchmarking, warm-up and accelerator synchronization: https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html
- Scikit-learn evaluation definitions: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
- Avoiding data leakage: https://scikit-learn.org/stable/common_pitfalls.html

The helper records installed versions for reproducibility. Test availability is not guaranteed merely because an import succeeds.